In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial
from steer_materials.CellMaterials.Base import TapeMaterial

from steer_opencell_design.Components.CurrentCollectors.Tabless import TablessCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.Laminate import Laminate
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.Constructions.ElectrodeAssemblies.WindingEquipment import RoundMandrel
from steer_opencell_design.Constructions.ElectrodeAssemblies.Tape import Tape

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data(table_name='cells', condition="name = 'Vendor G NFPP'")

In [4]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials',
 'tape_materials']

In [5]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")
tape_material = TapeMaterial.from_database("Kapton")

In [6]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=2269,
    coated_width=128,
    insulation_width=4,
    thickness=13
)

cathode_active_material = CathodeMaterial.from_database("NFPP")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=1.89,
    mass_loading=14.99,
    insulation_material=insulation_material,
    insulation_thickness=3
)



In [7]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=2269,
    coated_width=127,
    thickness=13
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.04,
    mass_loading=6.39
)

In [8]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=130,
    length=2640,
    thickness=12
)

bottom_separator = Separator(
    material=separator_material,
    width=130,
    length=2640,
    thickness=12
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator
)


In [9]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=10,
    length=500,
)

tape = Tape(
    material = tape_material,
    thickness=30
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    tape=tape,
    additional_tape_wraps=3,
    name="Vendor G NFPP",
)

In [10]:
my_jellyroll.roll_properties

,Turns
Anode A Side Coating Turns,33.55
Anode Current Collector Turns,33.55
Anode B Side Coating Turns,33.55
Cathode A Side Coating Turns,33.55
Cathode Current Collector Turns,33.55
Cathode B Side Coating Turns,33.55
Bottom Separator Turns,41.32
Bottom Separator Inner Turns,5.90
Bottom Separator Outer Turns,1.83
Top Separator Turns,41.32


In [11]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [12]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [13]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,CBAK-32140NS,gASVxAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:06,0.4.1,Na/Na+
1,Cell 2,gASVGA8BAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:09,0.4.1,Na/Na+
2,Cell 4,gASVXwkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:11,0.4.1,Na/Na+
3,HiNa-NaCR32140-MP10,gASVxAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:13,0.4.1,Na/Na+
4,QNAS-S40160NL,gASVxAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:16,0.4.1,Na/Na+
5,QNAS-S40160RL,gASVxAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:18,0.4.1,Na/Na+
6,UniGrid-NCO32140,gASVCCEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:20,0.4.1,Na/Na+
7,Vendor D NFPP,gASVGwABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-13 15:23:22,0.4.1,Na/Na+
8,Vendor E NFPP,gASVKQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-13 15:23:24,0.4.1,Na/Na+
9,Vendor G NFM,gASVxAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-13 15:23:27,0.4.1,Na/Na+
